# CSI Signal Processing and Feature Engineering

This notebook audits the WiCount CSI archive, compares DSP filters, evaluates normalization strategies, builds windowed feature matrices, and stores the outputs in `analysis/processed` and `analysis/outputs`.

Scope: all 15 CSV recordings from the original archive, without modifying the raw dataset and without any ML training.

In [1]:
import ast
import json
import math
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import median_filter
from scipy.signal import savgol_filter

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "PRD.md").is_file() and (path / "analysis").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate WiFiSENSE project root from notebook working directory")

root = PROJECT_ROOT
zip_path = root / "analysis" / "dataset" / "WiCount" / "WiCount.zip"
out_dir = root / "analysis" / "outputs"
processed_dir = root / "analysis" / "processed"
(signal_dir := out_dir / "signal_filtering").mkdir(parents=True, exist_ok=True)
(feature_dir := out_dir / "feature_analysis").mkdir(parents=True, exist_ok=True)
(window_dir := out_dir / "windowing").mkdir(parents=True, exist_ok=True)
(processed_features_dir := processed_dir / "features").mkdir(parents=True, exist_ok=True)
(processed_windows_dir := processed_dir / "windows").mkdir(parents=True, exist_ok=True)

print(f"ZIP exists: {zip_path.exists()}")
print(f"Project root: {root}")


ZIP exists: True
Project root: G:\My Drive\github\projects\WiFiSANSE


In [2]:
def read_csv_payloads(zip_path: Path):
    import zipfile
    results = []
    with zipfile.ZipFile(zip_path) as z:
        files = sorted([n for n in z.namelist() if n.lower().endswith('.csv')])
        for file_name in files:
            occupancy = Path(file_name).parts[-2]
            stem = Path(file_name).stem
            recording_id = stem
            with z.open(file_name) as fh:
                df = pd.read_csv(fh)
            rows = []
            invalid_rows = 0
            for _, row in df.iterrows():
                try:
                    vals = ast.literal_eval(row["data"])
                except Exception:
                    invalid_rows += 1
                    continue
                if not isinstance(vals, list) or len(vals) != 104:
                    invalid_rows += 1
                    continue
                arr = np.asarray(vals, dtype=float)
                if arr.size != 104 or not np.all(np.isfinite(arr)):
                    invalid_rows += 1
                    continue
                real = arr[0::2]
                imag = arr[1::2]
                if real.size != 52 or imag.size != 52:
                    invalid_rows += 1
                    continue
                amplitude = np.sqrt(real ** 2 + imag ** 2)
                rows.append({
                    "recording_id": recording_id,
                    "occupancy": int(occupancy),
                    "seq": int(row.get("seq", len(rows))),
                    "timestamp": row.get("timestamp"),
                    "rssi": row.get("rssi"),
                    "local_timestamp": row.get("local_timestamp"),
                    "amplitude": amplitude,
                    "data_len": len(vals),
                })
            amplitude_matrix = np.vstack([r['amplitude'] for r in rows]) if rows else np.empty((0, 52), dtype=float)
            results.append({
                "file_name": file_name,
                "recording_id": recording_id,
                "occupancy": int(occupancy),
                "df": df,
                "rows": rows,
                "invalid_rows": invalid_rows,
                "valid_rows": len(rows),
                "amplitude_matrix": amplitude_matrix,
            })
    return results

recordings = read_csv_payloads(zip_path)
print(f"Loaded {len(recordings)} recordings")
print("Valid rows per recording:", [r['valid_rows'] for r in recordings])
print("Invalid rows per recording:", [r['invalid_rows'] for r in recordings])
summary = pd.DataFrame([
    {
        "recording_id": r["recording_id"],
        "occupancy": r["occupancy"],
        "file_name": r["file_name"],
        "valid_rows": r["valid_rows"],
        "invalid_rows": r["invalid_rows"],
        "shape": r["amplitude_matrix"].shape,
    }
    for r in recordings
])
print(summary.head())
summary.to_csv(processed_dir / "recording_summary.csv", index=False)


Loaded 15 recordings
Valid rows per recording: [5218, 4780, 5189, 5234, 5163, 5421, 5644, 5128, 5376, 5307, 5141, 5467, 5221, 5423, 4906]
Invalid rows per recording: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
                    recording_id  occupancy  \
0  2023-12-21_19-23-53-985_104_1          0   
1  2023-12-21_19-27-21-930_104_1          1   
2  2023-12-21_19-31-17-706_104_1          1   
3  2023-12-21_19-34-42-783_104_1          1   
4  2023-12-21_19-37-36-861_104_1          1   

                                  file_name  valid_rows  invalid_rows  \
0  data/0/2023-12-21_19-23-53-985_104_1.csv        5218             0   
1  data/1/2023-12-21_19-27-21-930_104_1.csv        4780             0   
2  data/1/2023-12-21_19-31-17-706_104_1.csv        5189             0   
3  data/1/2023-12-21_19-34-42-783_104_1.csv        5234             0   
4  data/1/2023-12-21_19-37-36-861_104_1.csv        5163             0   

        shape  
0  (5218, 52)  
1  (4780, 52)  
2  (5189, 52)  
3 

## Filtering experiment

Filter comparisons use the original amplitude matrix on each recording, preserving all 52 subcarriers. Metrics are computed on the full set of valid CSI amplitude values for each method; they are compared against the raw amplitude baseline, not against a single plot alone.

In [3]:
def ma_filter(x, window):
    if len(x) < 2:
        return np.asarray(x, dtype=float).copy()
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(x, kernel, mode="same")

def ema_filter(x, alpha):
    y = np.asarray(x, dtype=float).copy()
    if y.size == 0:
        return y
    y[0] = x[0]
    for i in range(1, y.size):
        y[i] = alpha * x[i] + (1.0 - alpha) * y[i-1]
    return y

def apply_filter_to_matrix(matrix, method_name, **params):
    matrix = np.asarray(matrix, dtype=float)
    if matrix.size == 0:
        return matrix.copy()
    if method_name == "raw":
        return matrix.copy()
    if method_name.startswith("moving_average"):
        window = params["window"]
        out = np.apply_along_axis(lambda s: ma_filter(s, window), 0, matrix)
        return out
    if method_name.startswith("savitzky_golay"):
        window = params["window"]
        poly = params.get("polyorder", 2)
        if window >= matrix.shape[0]:
            window = max(5, (matrix.shape[0] // 2) * 2 + 1)
        if window % 2 == 0:
            window += 1
        if window > matrix.shape[0]:
            window = matrix.shape[0] if matrix.shape[0] % 2 == 1 else matrix.shape[0] - 1
        if window < 3:
            window = 3
        out = np.apply_along_axis(lambda s: savgol_filter(s, window_length=window, polyorder=poly, mode="interp"), 0, matrix)
        return out
    if method_name.startswith("median_filter"):
        size = params["kernel"]
        return median_filter(matrix, size=(size, 1), mode="nearest")
    if method_name.startswith("ema"):
        alpha = params["alpha"]
        out = np.apply_along_axis(lambda s: ema_filter(s, alpha), 0, matrix)
        return out
    raise ValueError(f"Unknown method: {method_name}")

methods = [
    ("raw", {}),
    ("moving_average_3", {"window": 3}),
    ("moving_average_5", {"window": 5}),
    ("moving_average_7", {"window": 7}),
    ("moving_average_11", {"window": 11}),
    ("savitzky_golay_5", {"window": 5, "polyorder": 2}),
    ("savitzky_golay_7", {"window": 7, "polyorder": 2}),
    ("savitzky_golay_11", {"window": 11, "polyorder": 2}),
    ("median_filter_3", {"kernel": 3}),
    ("median_filter_5", {"kernel": 5}),
    ("median_filter_7", {"kernel": 7}),
    ("ema_0_1", {"alpha": 0.1}),
    ("ema_0_3", {"alpha": 0.3}),
    ("ema_0_5", {"alpha": 0.5}),
]

filter_rows = []
for rec in recordings:
    raw = rec["amplitude_matrix"]
    if raw.size == 0:
        continue
    for method_name, params in methods:
        filtered = apply_filter_to_matrix(raw, method_name, **params)
        diff = filtered - raw
        temporal_std = np.std(filtered, axis=0, ddof=1)
        row = {
            "recording_id": rec["recording_id"],
            "occupancy": rec["occupancy"],
            "method": method_name,
            "mean": float(filtered.mean()),
            "std": float(filtered.std(ddof=1)) if filtered.size > 1 else 0.0,
            "variance": float(filtered.var(ddof=1)) if filtered.size > 1 else 0.0,
            "min": float(filtered.min()),
            "max": float(filtered.max()),
            "range": float(filtered.max() - filtered.min()),
            "temporal_std_mean": float(np.mean(temporal_std)) if temporal_std.size else 0.0,
            "temporal_std_median": float(np.median(temporal_std)) if temporal_std.size else 0.0,
            "mae_vs_raw": float(np.mean(np.abs(diff))),
            "rms_vs_raw": float(np.sqrt(np.mean(diff ** 2))),
            "n_values": int(filtered.size),
        }
        filter_rows.append(row)

filter_df = pd.DataFrame(filter_rows)
filter_df.to_csv(processed_features_dir / "filter_comparison.csv", index=False)
print(filter_df.head())
method_summary = filter_df.groupby("method").agg({
    "mean": "mean",
    "std": "mean",
    "variance": "mean",
    "range": "mean",
    "temporal_std_median": "mean",
    "mae_vs_raw": "mean",
    "rms_vs_raw": "mean",
    "n_values": "sum",
}).reset_index()
print(method_summary.sort_values("mae_vs_raw"))


                    recording_id  occupancy             method       mean  \
0  2023-12-21_19-23-53-985_104_1          0                raw  14.312227   
1  2023-12-21_19-23-53-985_104_1          0   moving_average_3  14.310383   
2  2023-12-21_19-23-53-985_104_1          0   moving_average_5  14.308914   
3  2023-12-21_19-23-53-985_104_1          0   moving_average_7  14.307677   
4  2023-12-21_19-23-53-985_104_1          0  moving_average_11  14.305096   

        std   variance       min        max      range  temporal_std_mean  \
0  5.458830  29.798823  2.236068  30.265492  28.029424           1.047330   
1  5.387952  29.030025  3.366340  29.311793  25.945453           0.627403   
2  5.373873  28.878509  3.431101  28.639955  25.208854           0.508406   
3  5.367931  28.814682  3.200560  28.251593  25.051034           0.450755   
4  5.362742  28.759003  3.044244  28.063307  25.019063           0.400195   

   temporal_std_median  mae_vs_raw  rms_vs_raw  n_values  
0             0

In [4]:
# Visualize the filter comparison for one representative recording
rec = recordings[0]
raw = rec["amplitude_matrix"]
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
subcarrier_index = 0
x = np.arange(raw.shape[0])[:800]
axes[0].plot(x, raw[:800, subcarrier_index], label="raw", linewidth=1.5)
for method_name, params in [
    ("moving_average_3", {"window": 3}),
    ("moving_average_5", {"window": 5}),
    ("savitzky_golay_5", {"window": 5, "polyorder": 2}),
    ("median_filter_5", {"kernel": 5}),
    ("ema_0_3", {"alpha": 0.3}),
]:
    filtered = apply_filter_to_matrix(raw, method_name, **params)
    axes[0].plot(x, filtered[:800, subcarrier_index], label=method_name, alpha=0.8)
axes[0].set_title(f"Representative CSI amplitude time series (recording {rec['recording_id']}, subcarrier 0)")
axes[0].set_ylabel("Amplitude")
axes[0].legend(ncol=3, fontsize=8)
axes[0].grid(alpha=0.2)

method_order = ["raw", "moving_average_3", "moving_average_5", "moving_average_7", "moving_average_11", "savitzky_golay_5", "savitzky_golay_7", "savitzky_golay_11", "median_filter_3", "median_filter_5", "median_filter_7", "ema_0_1", "ema_0_3", "ema_0_5"]
metric_df = filter_df[filter_df["recording_id"] == rec["recording_id"]].copy()
metric_df = metric_df[metric_df["method"].isin(method_order)]
metric_df["method"] = pd.Categorical(metric_df["method"], categories=method_order, ordered=True)
metric_df = metric_df.sort_values("method")
axes[1].bar(metric_df["method"], metric_df["mae_vs_raw"], color="steelblue")
axes[1].set_title("Mean absolute error against raw amplitude")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
fig.savefig(signal_dir / "filter_methods_comparison.png", dpi=200)
plt.close(fig)
print("Saved:", signal_dir / "filter_methods_comparison.png")


Saved: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\signal_filtering\filter_methods_comparison.png


## Normalization experiments

The normalization section is descriptive only. In a future modeling stage, the normalization parameters must be fit on the training set alone to avoid leakage; this notebook documents the method and keeps the current scope explicit.

In [5]:
def zscore_norm(matrix):
    matrix = np.asarray(matrix, dtype=float)
    if matrix.size == 0:
        return matrix.copy()
    mean = matrix.mean(axis=0, keepdims=True)
    std = matrix.std(axis=0, ddof=1, keepdims=True)
    std[std == 0] = 1.0
    return (matrix - mean) / std

def minmax_norm(matrix):
    matrix = np.asarray(matrix, dtype=float)
    if matrix.size == 0:
        return matrix.copy()
    minv = matrix.min(axis=0, keepdims=True)
    maxv = matrix.max(axis=0, keepdims=True)
    rng = maxv - minv
    rng[rng == 0] = 1.0
    return (matrix - minv) / rng

norm_rows = []
for rec in recordings:
    raw = rec["amplitude_matrix"]
    if raw.size == 0:
        continue
    for mode in ["zscore", "minmax"]:
        normed = zscore_norm(raw) if mode == "zscore" else minmax_norm(raw)
        subcarrier_stats = np.asarray([np.mean(normed[:, i]) for i in range(normed.shape[1])])
        norm_rows.append({
            "recording_id": rec["recording_id"],
            "occupancy": rec["occupancy"],
            "method": mode,
            "level": "subcarrier",
            "mean": float(np.mean(normed)),
            "std": float(np.std(normed, ddof=1)) if normed.size > 1 else 0.0,
            "min": float(np.min(normed)),
            "max": float(np.max(normed)),
            "subcarrier_mean_mean": float(np.mean(subcarrier_stats)),
            "subcarrier_mean_std": float(np.std(subcarrier_stats, ddof=1)) if subcarrier_stats.size > 1 else 0.0,
        })
        wsize = 50
        stride = wsize // 2
        for start in range(0, raw.shape[0] - wsize + 1, stride):
            window = raw[start:start + wsize]
            n = zscore_norm(window) if mode == "zscore" else minmax_norm(window)
            norm_rows.append({
                "recording_id": rec["recording_id"],
                "occupancy": rec["occupancy"],
                "method": mode,
                "level": "window_50",
                "mean": float(np.mean(n)),
                "std": float(np.std(n, ddof=1)) if n.size > 1 else 0.0,
                "min": float(np.min(n)),
                "max": float(np.max(n)),
                "subcarrier_mean_mean": float(np.mean(n.mean(axis=0))),
                "subcarrier_mean_std": float(np.std(n.mean(axis=0), ddof=1)) if n.shape[1] > 1 else 0.0,
            })

norm_df = pd.DataFrame(norm_rows)
print(norm_df.head())
norm_df.to_csv(processed_features_dir / "normalization_summary.csv", index=False)

mean_plot = norm_df.groupby(["method", "level"], as_index=False)[["mean", "std", "min", "max"]].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for level, ax in zip(["subcarrier", "window_50"], axes):
    subset = mean_plot[mean_plot["level"] == level].copy()
    x = np.arange(len(subset))
    ax.bar(x - 0.2, subset["mean"], width=0.35, label="mean")
    ax.bar(x + 0.2, subset["std"], width=0.35, label="std")
    ax.set_xticks(x)
    ax.set_xticklabels(subset["method"], rotation=20)
    ax.set_title(f"Normalization summary at {level} level")
    ax.legend()
plt.tight_layout()
fig.savefig(feature_dir / "normalization_summary.png", dpi=200)
plt.close(fig)
print("Saved:", feature_dir / "normalization_summary.png")


                    recording_id  occupancy  method       level          mean  \
0  2023-12-21_19-23-53-985_104_1          0  zscore  subcarrier  1.076299e-14   
1  2023-12-21_19-23-53-985_104_1          0  zscore   window_50 -1.399223e-15   
2  2023-12-21_19-23-53-985_104_1          0  zscore   window_50 -1.486674e-15   
3  2023-12-21_19-23-53-985_104_1          0  zscore   window_50 -1.585057e-15   
4  2023-12-21_19-23-53-985_104_1          0  zscore   window_50 -8.417199e-16   

        std        min        max  subcarrier_mean_mean  subcarrier_mean_std  
0  0.999906 -10.260598  13.771722          1.075945e-14         1.593752e-13  
1  0.990140  -2.656056   3.153688         -1.419260e-15         3.171286e-15  
2  0.990140  -2.657099   2.964359         -1.487069e-15         3.154411e-15  
3  0.990140  -2.684097   3.125056         -1.581366e-15         3.428846e-15  
4  0.990140  -3.004216   2.605152         -8.197236e-16         2.729899e-15  


Saved: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\feature_analysis\normalization_summary.png


## Windowing experiment

Window sizes 25, 50, 100, and 128 are evaluated. Stride is fixed at half the window length to enforce a 50% overlap. Every window is constrained to remain inside a single recording and carries occupancy and recording metadata.

In [6]:
def make_windows_for_recording(recording, window_size):
    matrix = recording["amplitude_matrix"]
    if matrix.size == 0:
        return [], 0
    stride = window_size // 2
    windows = []
    n = matrix.shape[0]
    last_end = 0
    for start in range(0, n - window_size + 1, stride):
        end = start + window_size
        segment = matrix[start:end]
        windows.append({
            "recording_id": recording["recording_id"],
            "occupancy": recording["occupancy"],
            "window_size": window_size,
            "stride": stride,
            "start_index": int(start),
            "end_index": int(end),
            "row_count": int(segment.shape[0]),
            "data": segment,
            "window_id": f"{recording['recording_id']}_{window_size}_{start}",
        })
        last_end = end
    remainder = n - last_end
    return windows, remainder

window_sizes = [25, 50, 100, 128]
all_window_rows = []
for w in window_sizes:
    win_rows = []
    per_rec = []
    for rec in recordings:
        windows, remainder = make_windows_for_recording(rec, w)
        per_rec.append({
            "recording_id": rec["recording_id"],
            "occupancy": rec["occupancy"],
            "window_size": w,
            "n_windows": len(windows),
            "remainder_samples": int(remainder),
        })
        for win in windows:
            win_rows.append({
                "window_id": win["window_id"],
                "recording_id": win["recording_id"],
                "occupancy": win["occupancy"],
                "window_size": win["window_size"],
                "stride": win["stride"],
                "start_index": win["start_index"],
                "end_index": win["end_index"],
                "row_count": win["row_count"],
            })
    pd.DataFrame(win_rows).to_csv(processed_windows_dir / f"window_summary_{w}.csv", index=False)
    pd.DataFrame(per_rec).to_csv(processed_windows_dir / f"window_recording_summary_{w}.csv", index=False)
    all_window_rows.extend(per_rec)

window_summary = pd.DataFrame(all_window_rows)
window_summary.to_csv(processed_windows_dir / "window_summary_all_sizes.csv", index=False)
print(window_summary.head())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for size in window_sizes:
    subset = window_summary[window_summary["window_size"] == size]
    counts = subset.groupby("occupancy")["n_windows"].sum().sort_index()
    axes[0].bar([f"{size}:{k}" for k in counts.index], counts.values, alpha=0.8)
axes[0].set_title("Window count by occupancy and size")
axes[0].set_ylabel("Window count")
axes[0].tick_params(axis='x', rotation=45)

by_size = window_summary.groupby("window_size")["n_windows"].sum()
axes[1].bar(by_size.index.astype(str), by_size.values, color="darkorange")
axes[1].set_title("Total window count by size")
axes[1].set_ylabel("Total windows")
plt.tight_layout()
fig.savefig(window_dir / "window_counts.png", dpi=200)
plt.close(fig)
print("Saved:", window_dir / "window_counts.png")


                    recording_id  occupancy  window_size  n_windows  \
0  2023-12-21_19-23-53-985_104_1          0           25        433   
1  2023-12-21_19-27-21-930_104_1          1           25        397   
2  2023-12-21_19-31-17-706_104_1          1           25        431   
3  2023-12-21_19-34-42-783_104_1          1           25        435   
4  2023-12-21_19-37-36-861_104_1          1           25        429   

   remainder_samples  
0                  9  
1                  3  
2                  4  
3                  1  
4                  2  


Saved: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\windowing\window_counts.png


In [7]:
def compute_feature_vector(window_matrix):
    window_matrix = np.asarray(window_matrix, dtype=float)
    if window_matrix.size == 0:
        return {}

    n_samples, n_subs = window_matrix.shape
    sub_mean = window_matrix.mean(axis=0)
    sub_std = window_matrix.std(axis=0, ddof=1)
    sub_var = window_matrix.var(axis=0, ddof=1)
    sub_min = window_matrix.min(axis=0)
    sub_max = window_matrix.max(axis=0)
    sub_range = sub_max - sub_min
    q1 = np.quantile(window_matrix, 0.25, axis=0)
    q3 = np.quantile(window_matrix, 0.75, axis=0)
    iqr = q3 - q1
    centered = window_matrix - sub_mean
    mad = np.mean(np.abs(centered), axis=0)
    rms = np.sqrt(np.mean(window_matrix ** 2, axis=0))
    energy = np.sum(window_matrix ** 2, axis=0)
    diff = np.diff(window_matrix, axis=0)
    if n_samples > 1:
        zcr = np.mean((centered[:-1] * centered[1:]) < 0.0, axis=0)
        first_diff_mean = np.mean(diff, axis=0)
        first_diff_std = np.std(diff, axis=0, ddof=1)
    else:
        zcr = np.zeros(n_subs)
        first_diff_mean = np.zeros(n_subs)
        first_diff_std = np.zeros(n_subs)

    skew = np.zeros(n_subs, dtype=float)
    kurt = np.zeros(n_subs, dtype=float)
    valid_std = sub_std > 0
    if n_samples > 2 and np.any(valid_std):
        centered_valid = centered[:, valid_std]
        denom = (n_samples - 1) * (sub_std[valid_std] ** 3)
        skew[valid_std] = np.sum(centered_valid ** 3, axis=0) / denom
    if n_samples > 3 and np.any(valid_std):
        centered_valid = centered[:, valid_std]
        denom = (n_samples - 1) * (sub_std[valid_std] ** 4)
        kurt[valid_std] = np.sum(centered_valid ** 4, axis=0) / denom - 3.0

    corr = np.corrcoef(window_matrix, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    upper = corr[np.triu_indices_from(corr, 1)]

    features = {}
    for idx in range(n_subs):
        features.update({
            f"sub_{idx}_mean": float(sub_mean[idx]),
            f"sub_{idx}_median": float(np.median(window_matrix[:, idx])),
            f"sub_{idx}_std": float(sub_std[idx]),
            f"sub_{idx}_var": float(sub_var[idx]),
            f"sub_{idx}_min": float(sub_min[idx]),
            f"sub_{idx}_max": float(sub_max[idx]),
            f"sub_{idx}_range": float(sub_range[idx]),
            f"sub_{idx}_q1": float(q1[idx]),
            f"sub_{idx}_q3": float(q3[idx]),
            f"sub_{idx}_iqr": float(iqr[idx]),
            f"sub_{idx}_skewness": float(np.nan_to_num(skew[idx], nan=0.0, posinf=0.0, neginf=0.0)),
            f"sub_{idx}_kurtosis": float(np.nan_to_num(kurt[idx], nan=0.0, posinf=0.0, neginf=0.0)),
            f"sub_{idx}_mad": float(mad[idx]),
            f"sub_{idx}_temporal_std": float(sub_std[idx]),
            f"sub_{idx}_rms": float(rms[idx]),
            f"sub_{idx}_energy": float(energy[idx]),
            f"sub_{idx}_zero_crossing_rate": float(zcr[idx]),
            f"sub_{idx}_first_diff_mean": float(first_diff_mean[idx]),
            f"sub_{idx}_first_diff_std": float(first_diff_std[idx]),
        })

    features["cross_subcarrier_mean_amp"] = float(sub_mean.mean())
    features["cross_subcarrier_std_amp"] = float(sub_mean.std(ddof=1)) if sub_mean.size > 1 else 0.0
    features["cross_subcarrier_var_amp"] = float(sub_mean.var(ddof=1)) if sub_mean.size > 1 else 0.0
    features["cross_subcarrier_corr_median"] = float(np.median(upper)) if upper.size else 0.0
    return features


In [8]:
feature_frames = {}
for w in window_sizes:
    rows = []
    for rec in recordings:
        windows, _ = make_windows_for_recording(rec, w)
        for win in windows:
            feat = compute_feature_vector(win["data"])
            feat.update({
                "window_id": win["window_id"],
                "recording_id": win["recording_id"],
                "occupancy": win["occupancy"],
                "window_size": w,
                "start_index": win["start_index"],
                "end_index": win["end_index"],
                "row_count": win["row_count"],
            })
            rows.append(feat)
    feature_df = pd.DataFrame(rows)
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    feature_frames[w] = feature_df
    feature_df.to_csv(processed_features_dir / f"feature_matrix_{w}.csv", index=False)
    print(f"Saved feature matrix for window {w}: {feature_df.shape}")

print(feature_frames[50].head())


Saved feature matrix for window 25: (6529, 999)


Saved feature matrix for window 50: (3122, 999)


Saved feature matrix for window 100: (1550, 999)


Saved feature matrix for window 128: (1206, 999)
   sub_0_mean  sub_0_median  sub_0_std  sub_0_var  sub_0_min  sub_0_max  \
0    8.928350      9.055385   0.673521   0.453631   7.810250  10.770330   
1    8.930586      9.055385   0.759053   0.576161   7.615773  10.630146   
2    8.766018      8.773299   0.818736   0.670329   7.211103  10.630146   
3    8.678619      8.602325   0.701003   0.491406   7.071068  10.198039   
4    8.733907      8.944272   0.768561   0.590686   7.071068  10.440307   

   sub_0_range  sub_0_q1  sub_0_q3  sub_0_iqr  ...  cross_subcarrier_std_amp  \
0     2.960080  8.485281  9.219544   0.734263  ...                  5.403538   
1     3.014373  8.246211  9.380372   1.134161  ...                  5.403327   
2     3.419043  8.062258  9.219544   1.157287  ...                  5.329124   
3     3.126971  8.246211  9.219544   0.973333  ...                  5.423950   
4     3.369239  8.305979  9.219544   0.913566  ...                  5.482168   

   cross_subcarrier

In [9]:
feature_quality = []
for w, frame in feature_frames.items():
    numeric = frame.select_dtypes(include=[np.number])
    feature_quality.append({
        "window_size": w,
        "n_windows": int(frame.shape[0]),
        "n_features": int(frame.shape[1] - 7),
        "n_nan": int(frame.isna().sum().sum()),
        "n_inf": int(np.isinf(numeric.to_numpy(dtype=float)).sum()),
        "class_counts": str(frame.groupby("occupancy").size().to_dict()),
    })
feature_quality_df = pd.DataFrame(feature_quality)
print(feature_quality_df)
feature_quality_df.to_csv(processed_features_dir / "feature_quality_summary.csv", index=False)

feature_subset = feature_frames[50][[
    "sub_0_mean", "sub_1_mean", "sub_2_mean", "sub_10_mean", "sub_20_mean", "sub_30_mean", "sub_40_mean", "cross_subcarrier_mean_amp"
]].copy()
heatmap = feature_subset.corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heatmap, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(heatmap.columns)))
ax.set_yticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns, rotation=30, ha="right")
ax.set_yticklabels(heatmap.columns)
ax.set_title("Feature correlation heatmap (window size 50)")
fig.colorbar(im, ax=ax, label="Pearson correlation")
fig.tight_layout()
fig.savefig(feature_dir / "feature_correlation_heatmap.png", dpi=200)
plt.close(fig)
print("Saved:", feature_dir / "feature_correlation_heatmap.png")

selected_feature = "cross_subcarrier_mean_amp"
fig, ax = plt.subplots(figsize=(8, 5))
for occ in sorted(feature_frames[50]["occupancy"].unique()):
    vals = feature_frames[50][feature_frames[50]["occupancy"] == occ][selected_feature]
    ax.hist(vals.values, bins=25, alpha=0.5, label=f"occupancy={occ}")
ax.set_title(f"Distribution of {selected_feature} by occupancy")
ax.set_xlabel(selected_feature)
ax.set_ylabel("Window count")
ax.legend()
fig.tight_layout()
fig.savefig(feature_dir / "feature_distribution_by_class.png", dpi=200)
plt.close(fig)
print("Saved:", feature_dir / "feature_distribution_by_class.png")

# Compute PCA manually (NumPy SVD) to keep dependencies minimal and avoid ML training commitments.
X = feature_frames[50][[c for c in feature_frames[50].columns if c not in ["window_id", "recording_id", "occupancy", "window_size", "start_index", "end_index", "row_count"]]].to_numpy(dtype=float)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_centered = X - X.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
components = U[:, :2] * S[:2]
fig, ax = plt.subplots(figsize=(8, 5))
for occ in sorted(feature_frames[50]["occupancy"].unique()):
    idx = feature_frames[50]["occupancy"] == occ
    ax.scatter(components[idx, 0], components[idx, 1], s=12, alpha=0.7, label=f"occupancy={occ}")
ax.set_title("PCA projection (exploratory only)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(feature_dir / "pca_exploration.png", dpi=200)
plt.close(fig)
print("Saved:", feature_dir / "pca_exploration.png")


   window_size  n_windows  n_features  n_nan  n_inf  \
0           25       6529         992      0      0   
1           50       3122         992      0      0   
2          100       1550         992      0      0   
3          128       1206         992      0      0   

                         class_counts  
0  {0: 433, 1: 3037, 2: 2652, 3: 407}  
1  {0: 207, 1: 1452, 2: 1268, 3: 195}  
2     {0: 103, 1: 720, 2: 630, 3: 97}  
3      {0: 80, 1: 561, 2: 490, 3: 75}  


Saved: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\feature_analysis\feature_correlation_heatmap.png


Saved: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\feature_analysis\feature_distribution_by_class.png


Saved: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\feature_analysis\pca_exploration.png


In [10]:
window_df_all = pd.concat([
    pd.read_csv(processed_windows_dir / f"window_recording_summary_{w}.csv")
    for w in window_sizes
], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for occ, ax in zip(sorted(window_df_all["occupancy"].unique()), axes):
    subset = window_df_all[window_df_all["occupancy"] == occ]
    means = subset.groupby("window_size")["n_windows"].mean()
    ax.bar(means.index.astype(str), means.values, color="tab:green")
    ax.set_title(f"Occupancy {occ}: average windows per recording")
    ax.set_xlabel("Window size")
    ax.set_ylabel("Average windows")
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(window_dir / "window_recording_variation.png", dpi=200)
plt.close(fig)
print("Saved:", window_dir / "window_recording_variation.png")


Saved: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\windowing\window_recording_variation.png


In [11]:
required_paths = [
    processed_features_dir / "filter_comparison.csv",
    processed_features_dir / "normalization_summary.csv",
    processed_features_dir / "feature_quality_summary.csv",
    processed_windows_dir / "window_summary_all_sizes.csv",
    feature_dir / "feature_correlation_heatmap.png",
    feature_dir / "feature_distribution_by_class.png",
    feature_dir / "pca_exploration.png",
    signal_dir / "filter_methods_comparison.png",
    window_dir / "window_counts.png",
    window_dir / "window_recording_variation.png",
]
for p in required_paths:
    print(f"{p.name}: exists={p.exists()} size={p.stat().st_size if p.exists() else 0}")

all_valid = True
for w, df in feature_frames.items():
    numeric = df.select_dtypes(include=[np.number])
    has_inf = bool(numeric.size and np.isinf(numeric.to_numpy(dtype=float)).any())
    if df.isna().any().any() or has_inf:
        all_valid = False
        print(f"Invalid values detected for window={w}")
print(f"Feature matrices valid: {all_valid}")
print(f"Total recordings processed: {len(recordings)}")
print(f"Total rows in all feature matrices: {sum(df.shape[0] for df in feature_frames.values())}")


filter_comparison.csv: exists=True size=48964
normalization_summary.csv: exists=True size=977652
feature_quality_summary.csv: exists=True size=270
window_summary_all_sizes.csv: exists=True size=2633
feature_correlation_heatmap.png: exists=True size=108248
feature_distribution_by_class.png: exists=True size=58514
pca_exploration.png: exists=True size=379666
filter_methods_comparison.png: exists=True size=541385
window_counts.png: exists=True size=79981
window_recording_variation.png: exists=True size=52339
Feature matrices valid: True
Total recordings processed: 15
Total rows in all feature matrices: 12407


## Conclusion

This notebook validates the DSP and feature engineering workflow on the original WiCount archive, stores the feature windows and filtered comparisons, and records the exact assumptions used. It stops at preprocessing and feature engineering, without moving into ML training.